# Landing → Bronze

Ingestão dos dados brutos do projeto Preço na Bomba.

- **Preços de combustíveis por revenda (ANP):** arquivos semestrais baixados do portal de dados abertos e carregados de forma incremental com Auto Loader
- **IPCA mensal (API SGS do Banco Central):** usado na Gold para corrigir os preços pela inflação

## 1. Parâmetros

In [0]:
catalogo = "preco_na_bomba"
bronze = f"{catalogo}.bronze"
volume_path = f"/Volumes/{catalogo}/bronze/inputs"
pasta_precos = f"{volume_path}/precos"
pasta_checkpoints = f"{volume_path}/_checkpoints"

## 2. Catálogo, database e Volume

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalogo}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze}.inputs")

DataFrame[]

## 3. Landing: download dos semestres da ANP

A ANP publica um arquivo por semestre, compactado em `.zip`. Cada semestre é baixado, extraído e salvo como CSV no Volume `inputs/precos`.

- Os semestres são definidos no widget `semestres` (formato `AAAA-SS`)
- Semestres que já estão no Volume não são baixados de novo

In [0]:
import io
import os
import zipfile
import requests

dbutils.widgets.text("semestres", "2024-01,2024-02,2025-01,2025-02", "Semestres (AAAA-SS)")
semestres = [s.strip() for s in dbutils.widgets.get("semestres").split(",")]

url_base = "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/shpc/dsas/ca"
os.makedirs(pasta_precos, exist_ok=True)

for semestre in semestres:
    destino = f"{pasta_precos}/ca-{semestre}.csv"

    if os.path.exists(destino):
        print(f"{semestre}: já está no Volume")
        continue

    resposta = requests.get(f"{url_base}/ca-{semestre}.zip", headers={"User-Agent": "Mozilla/5.0"}, timeout=300)
    resposta.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(resposta.content)) as arquivo_zip:
        nome_csv = [n for n in arquivo_zip.namelist() if n.lower().endswith(".csv")][0]
        with arquivo_zip.open(nome_csv) as origem, open(destino, "wb") as saida:
            saida.write(origem.read())

    print(f"{semestre}: {os.path.getsize(destino) / 1e6:.0f} MB")

2024-01: 81 MB
2024-02: 72 MB
2025-01: 72 MB
2025-02: 64 MB


## 4. Bronze: ingestão incremental com Auto Loader

O Auto Loader registra no checkpoint quais arquivos já foram processados, então cada execução lê apenas os CSVs novos da pasta `precos`.

- `trigger(availableNow=True)`: processa os arquivos pendentes e encerra, como uma carga em lote
- Todas as colunas entram como texto; a tipagem acontece na Silver
- Os nomes originais têm espaços e hífens, que o Delta não aceita, por isso são renomeados para snake_case já na leitura
- `arquivo_origem` e `ingestion_datetime` registram de onde e quando cada linha veio

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

colunas = [
    "regiao", "uf", "municipio", "revenda", "cnpj", "nome_rua", "numero_rua", "complemento",
    "bairro", "cep", "produto", "data_coleta", "valor_venda", "valor_compra", "unidade_medida", "bandeira",
]
schema_precos = StructType([StructField(c, StringType()) for c in colunas])

(
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("sep", ";")
    .schema(schema_precos)
    .load(pasta_precos)
    .withColumn("arquivo_origem", F.col("_metadata.file_name"))
    .withColumn("ingestion_datetime", F.current_timestamp())
    .writeStream
    .option("checkpointLocation", f"{pasta_checkpoints}/tb_precos_revenda")
    .trigger(availableNow=True)
    .toTable(f"{bronze}.tb_precos_revenda")
    .awaitTermination()
)

In [0]:
display(spark.sql(f"""
    SELECT arquivo_origem, COUNT(*) AS linhas
    FROM {bronze}.tb_precos_revenda
    GROUP BY arquivo_origem
    ORDER BY arquivo_origem
"""))

arquivo_origem,linhas
ca-2024-01.csv,477154
ca-2024-02.csv,421382
ca-2025-01.csv,429523
ca-2025-02.csv,384208
